# Test A/B

## **Descripción del ejercicio**

Has recibido una tarea analítica de una tienda en línea internacional. Tus predecesores no consiguieron completarla: lanzaron una prueba A/B y luego abandonaron (para iniciar una granja de sandías en Brasil). Solo dejaron las especificaciones técnicas y los resultados de las pruebas.

### 1) Objetivo del estudio

Meta del experimento: evaluar si el embudo nuevo (grupo B) mejora la conversión de usuarios en los primeros 14 días tras registrarse, en los pasos:

product_page → product_card → purchase

Hipótesis esperada del negocio: en cada etapa del embudo, el grupo B debería tener ≥ 10% de aumento relativo vs A.

### 2) Cargar datos y revisar tipos, nulos y duplicados

In [1]:
import pandas as pd
import numpy as np
from scipy import stats as st
from math import sqrt
from scipy.stats import norm

marketing = pd.read_csv('ab_project_marketing_events_us.csv')
new_users = pd.read_csv('final_ab_new_users_upd_us.csv')
events = pd.read_csv('final_ab_events_upd_us.csv')
participants = pd.read_csv('final_ab_participants_upd_us.csv')

marketing.info()
new_users.info()
events.info()
participants.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14 entries, 0 to 13
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   name       14 non-null     object
 1   regions    14 non-null     object
 2   start_dt   14 non-null     object
 3   finish_dt  14 non-null     object
dtypes: object(4)
memory usage: 580.0+ bytes
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 58703 entries, 0 to 58702
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   user_id     58703 non-null  object
 1   first_date  58703 non-null  object
 2   region      58703 non-null  object
 3   device      58703 non-null  object
dtypes: object(4)
memory usage: 1.8+ MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 423761 entries, 0 to 423760
Data columns (total 4 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   user_id     423761 non-null  obj

En el info() de los dataframe salen las fechas en tipo de dato objeto, lo vamos a convertir a tipo de dato fecha para el análisis correspondinte.

In [2]:
# Converciones a datetime
marketing['start_dt']  = pd.to_datetime(marketing['start_dt'])
marketing['finish_dt'] = pd.to_datetime(marketing['finish_dt'])

new_users['first_date'] = pd.to_datetime(new_users['first_date'])

events['event_dt'] = pd.to_datetime(events['event_dt'])

marketing.info()
new_users.info()
events.info()
participants.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14 entries, 0 to 13
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   name       14 non-null     object        
 1   regions    14 non-null     object        
 2   start_dt   14 non-null     datetime64[ns]
 3   finish_dt  14 non-null     datetime64[ns]
dtypes: datetime64[ns](2), object(2)
memory usage: 580.0+ bytes
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 58703 entries, 0 to 58702
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   user_id     58703 non-null  object        
 1   first_date  58703 non-null  datetime64[ns]
 2   region      58703 non-null  object        
 3   device      58703 non-null  object        
dtypes: datetime64[ns](1), object(3)
memory usage: 1.8+ MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 423761 entries, 0 to 423760
Data columns (total

La columna details contiene muchos valores nulos porque solo almacena información adicional para ciertos eventos (por ejemplo, el total del pedido en compras). Para eventos como vistas o carrito, este campo puede quedar vacío de forma esperada.


In [3]:
print("Duplicados completos marketing:", marketing.duplicated().sum())
print("Duplicados completos new_users:", new_users.duplicated().sum())
print("Duplicados completos events:", events.duplicated().sum())
print("Duplicados completos participants:", participants.duplicated().sum())

# Duplicados lógicos en events (mismo usuario, mismo evento, misma fecha-hora)
print("Duplicados lógicos events:",
    events.duplicated(subset=['user_id','event_dt','event_name']).sum())

Duplicados completos marketing: 0
Duplicados completos new_users: 0
Duplicados completos events: 0
Duplicados completos participants: 0
Duplicados lógicos events: 0


No se detectaron registros duplicados ni completos ni lógicos en ninguna tabla.

Esto indica:

	•El sistema de tracking no generó eventos repetidos.
	•No hay riesgo de inflar conversiones artificialmente.
	•No es necesario eliminar registros antes del análisis.

In [4]:
# Tamaño de los grupos en el test A/B
participants.query("ab_test == 'recommender_system_test'")['group'].value_counts()

group
A    2747
B     928
Name: count, dtype: int64

El experimento está fuertemente desbalanceado.

	•Grupo A: 2,747 usuarios
	•Grupo B: 928 usuarios

Este desbalance puede reducir potencia estadística, aumentar error estándar, hacer que el grupo B tenga resultados menos confiables

In [5]:
participants['ab_test'].value_counts()

ab_test
interface_eu_test          10850
recommender_system_test     3675
Name: count, dtype: int64

Esto significa que:

	•Hay dos pruebas A/B corriendo al mismo tiempo.
	•Nuestro test tiene 3,675 participantes totales.

Si un usuario está en ambas pruebas, puede afectar comportamiento.

In [6]:
tmp = participants.query("ab_test == 'recommender_system_test'")
(tmp.groupby('user_id')['group'].nunique() > 1).sum()

0

No hay usuarios asignados simultáneamente a A y B dentro del test.

El experimento está limpio en este punto.


In [7]:
tmp2 = tmp.merge(new_users, on='user_id', how='left')
tmp2['region'].value_counts()

region
EU           3481
N.America     119
APAC           45
CIS            30
Name: count, dtype: int64

El experimento estaba definido como:

15% de nuevos usuarios de la región EU

Pero encontramos:

	•119 usuarios de Norteamérica
	•45 de APAC
	•30 de CIS

Es decir:

3675 totales - 3481 EU = 194 usuarios fuera de EU

### Filtrar solo usuarios EU antes de hacer el análisis.

In [8]:
tmp_eu = tmp2[tmp2['region'] == 'EU'].copy()
tmp_eu['group'].value_counts()

group
A    2604
B     877
Name: count, dtype: int64

Durante la validación del experimento se detectó que, aunque el test estaba definido para usuarios de la región EU, 194 participantes pertenecían a otras regiones. Para garantizar la validez del análisis, estos usuarios fueron excluidos antes de evaluar los resultados.

Tras filtrar exclusivamente usuarios de la región EU, la muestra final quedó compuesta por 3,481 usuarios (2,604 en grupo A y 877 en grupo B).
Se observa un fuerte desbalance entre grupos (aproximadamente 75% vs 25%), así como un tamaño final inferior al planeado (6,000 usuarios). Esto puede afectar la potencia estadística del experimento.


In [9]:
# unir eventos con muestra EU
events_eu = events.merge(tmp_eu[['user_id','group','first_date']], on='user_id', how='inner')

# limitar a 14 días
events_eu['days'] = (events_eu['event_dt'] - events_eu['first_date']).dt.days
events_eu = events_eu[(events_eu['days'] >= 0) & (events_eu['days'] <= 14)]

funnel = ['product_page','product_card','purchase']

stage_users = (
    events_eu[events_eu['event_name'].isin(funnel)]
    .drop_duplicates(['user_id','event_name'])
    .groupby(['group','event_name'])['user_id']
    .nunique()
    .unstack(fill_value=0)
)

stage_users

event_name,product_page,purchase
group,,
A,1685,833
B,493,249


El grupo B (nuevo sistema de recomendaciones) presenta una tasa de conversión inferior al grupo A tanto en visitas a página de producto como en compras.

No se observa una mejora del 10% en ninguna etapa del embudo, como se había planteado en la hipótesis del experimento.

Además, no se detectó el evento product_card dentro del período analizado, lo que impide evaluar completamente el embudo definido originalmente. Esto podría indicar un problema de seguimiento de eventos o un cambio en la lógica del flujo de compra.



In [ ]:
def z_test(success_a, size_a, success_b, size_b, alpha=0.05):
    p1 = success_a / size_a
    p2 = success_b / size_b
    
    p_pool = (success_a + success_b) / (size_a + size_b)
    se = sqrt(p_pool * (1 - p_pool) * (1/size_a + 1/size_b))
    
    z = (p2 - p1) / se
    p_value = 2 * (1 - norm.cdf(abs(z)))
    
    print("Conversión A:", round(p1,4))
    print("Conversión B:", round(p2,4))
    print("z-stat:", round(z,4))
    print("p-value:", p_value)
    
    if p_value < alpha:
    if p2 > p1:
        print("Rechazamos H0: El grupo B tiene conversión significativamente mayor.")
    else:
        print("Rechazamos H0: El grupo B tiene conversión significativamente menor.")
else:
    print("No hay evidencia suficiente para afirmar diferencia significativa.")

# Product page
print("---- PRODUCT_PAGE ----")
z_test(1685, 2604, 493, 877)

# purchase
print("\n---- PURCHASE ----")
z_test(833, 2604, 249, 877)

---- PRODUCT_PAGE ----
Conversión A: 0.6471
Conversión B: 0.5621
z-stat: -4.4954
p-value: 6.942739359416805e-06
Rechazamos la hipótesis nula

---- PURCHASE ----
Conversión A: 0.3199
Conversión B: 0.2839
z-stat: -1.9906
p-value: 0.04652482738393027
Rechazamos la hipótesis nula


(0.31989247311827956,
 0.2839224629418472,
 -1.9906004806163913,
 0.04652482738393027)

### Evaluación estadística del A/B Test

Se realizó una prueba z de proporciones para comparar las tasas de conversión entre el grupo control (A) y el grupo experimental (B).

product_page

	•Conversión A: 64.71%
	•Conversión B: 56.21%
	•z = -4.50
	•p-value = 0.000007

La diferencia es estadísticamente significativa (p < 0.05).
El grupo B presenta una conversión significativamente menor que el grupo A.

purchase

	•Conversión A: 31.99%
	•Conversión B: 28.39%
	•z = -1.99
	•p-value = 0.0465

La diferencia también es significativa al nivel α = 0.05, aunque marginalmente.


## Conclusión General

El nuevo sistema de recomendaciones (grupo B) no solo no mejora el embudo, sino que reduce significativamente la conversión en la etapa de visualización de productos y muestra una tendencia negativa en compras.

No se cumple el objetivo esperado de un aumento del 10% en cada etapa del embudo.

Por lo tanto, con base en la evidencia estadística, no se recomienda implementar el nuevo sistema en producción sin realizar ajustes adicionales o una nueva iteración del experimento.